# Bronze Source Landscape

## Clinical Trial Intelligence Platform

### Objective

This notebook establishes the source landscape for the Bronze layer of the Clinical Trial Intelligence Platform before production ingestion.

The analysis is used to:

- identify all datasets delivered to the Amazon S3 landing zone,
- verify source-path accessibility,
- document source file naming and delivery patterns,
- understand file arrival cadence,
- and establish a complete inventory of datasets expected by the Bronze layer.

This notebook focuses only on the physical source landscape and delivery characteristics. Detailed per-feed schema and source profiling are performed in the corresponding source exploration notebooks, while ingestion strategy decisions are documented separately in the Bronze ingestion design notebook.

## 1. Source Landing Zone Configuration

### Question

What source systems and dataset families are delivered to the Amazon S3 landing zone?

### Purpose

Before evaluating individual files, the landing-zone structure is inspected to establish the physical organization of the source data available for Bronze ingestion.

This check identifies the top-level source families and confirms that the expected clinical data domains are accessible from Databricks.

In [0]:
landing_root = "s3://clinical-trial-intelligence-platform-sk/Landing/"
landing_items = dbutils.fs.ls(landing_root)
display(
    spark.createDataFrame(
        [   (
                item.name.rstrip("/"),
                item.path,
                "Directory" if item.isDir() else "File"
            )
            for item in landing_items
        ],
        ["source_family", "source_path", "object_type"]
    ).orderBy("source_family")
)

### Result

The Amazon S3 landing zone contains **seven top-level source families**:

- `CTMS`
- `EDC`
- `Lab`
- `Safety`
- `master`
- `protocol`
- `reference`

All seven source families are accessible from the Databricks environment and are organized as separate directories under the common `Landing/` prefix.

### Conclusion

The landing-zone connectivity check confirms that the expected clinical operational, protocol, master, and reference source families are available for Bronze-layer ingestion.

At this stage, no ingestion strategy is assigned. The next step is to inspect the datasets contained within each source family and establish a complete Bronze source inventory.

## 2. Bronze Dataset Inventory

### Question

What datasets are physically available within each source family in the S3 landing zone?

### Purpose

A complete dataset inventory is required to verify that every source expected by the Bronze layer is represented in the landing zone.

This check expands each top-level source family and records the immediate dataset-level directories or files available beneath it. The resulting inventory provides the source-level evidence for mapping landed datasets to Bronze tables in later design steps.

In [0]:
dataset_inventory = []
for family in landing_items:
    if family.isDir():
        for item in dbutils.fs.ls(family.path):
            dataset_inventory.append(
                (
                    family.name.rstrip("/"),
                    item.name.rstrip("/"),
                    item.path,
                    "Directory" if item.isDir() else "File"
                )
            )
dataset_inventory_df = spark.createDataFrame(
    dataset_inventory,
    [
        "source_family",
        "dataset",
        "source_path",
        "object_type"
    ]
)
display(
    dataset_inventory_df.orderBy(
        "source_family",
        "dataset"
    )
)

### Dataset Inventory Summary

The dataset-level inventory is summarized by source family to verify how many landed datasets or source objects are present under each top-level source domain.

In [0]:
from pyspark.sql import functions as F
inventory_summary_df = (
    dataset_inventory_df
    .groupBy("source_family")
    .agg(
        F.count("*").alias("dataset_count")
    )
    .orderBy("source_family")
)
display(inventory_summary_df)

In [0]:
total_datasets = dataset_inventory_df.count()
print(f"Total dataset-level objects discovered: {total_datasets}")

In [0]:
display(dataset_inventory_df.select(
        "source_family",
        "dataset",
        "object_type"
    )
    .orderBy(
        "source_family",
        "dataset"
    )
)

## 3. Source File Counts

### Question

How many source files are currently available for each dataset-level object in the landing zone?

### Purpose

File counts provide a first view of source delivery behaviour across the Bronze landing zone.

The comparison helps distinguish datasets delivered through multiple files over time from datasets represented by individual source files, without assigning an ingestion strategy at this stage.

In [0]:
file_count_results = []

for row in dataset_inventory_df.collect():

    source_family = row["source_family"]
    dataset = row["dataset"]
    source_path = row["source_path"]
    object_type = row["object_type"]

    if object_type == "Directory":
        files = [
            item
            for item in dbutils.fs.ls(source_path)
            if not item.isDir()
        ]
        file_count = len(files)

    else:
        # The dataset-level object itself is already a file
        file_count = 1

    file_count_results.append(
        (
            source_family,
            dataset,
            file_count
        )
    )

file_count_df = spark.createDataFrame(
    file_count_results,
    [
        "source_family",
        "dataset",
        "file_count"
    ]
)

display(
    file_count_df.orderBy(
        "source_family",
        "dataset"
    )
)